# HTG Algebra: Composing Hierarchical Tensor Graphs

This notebook introduces the core compositional operations of the `HierarchalTensorGraph` (HTG):
defining `Node` classes, wiring them with `add_edge`, renaming tensors across edges with
`rename`, and **nesting HTGs inside larger HTGs** to build multi-level hierarchies.

An HTG models computation as a **directed acyclic graph (DAG)** of callable nodes. Each node
receives a dict of named tensors and returns a dict of named tensors. Edges describe how one
node's outputs become another's inputs, with optional key renaming.

**What we build in this notebook:**

1. `a * (b + c)` — a two-node HTG (`Add` → `Multiply`)
2. `exp(a * (b + c))` — wrapping the two-node HTG inside a larger HTG as a child node
3. Inspecting graphs with `draw()` and `print_edge_labels()`

In [ ]:
%cd -q ..
from crest import HierarchalTensorGraph, Node
from math import exp

## Part 1 — Building `Node` Classes

A `Node` is a **basenode**: a single-operation unit that wraps a Python callable. You define:
- `node`: the callable (lambda or function) mapping a dict of input tensors to a dict of outputs
- `inputs`: `{key: tensor_spec}` — use `None` for any shape, or a NumPy array / `tf.TensorSpec`
- `outputs`: `{key: tensor_spec}` — same format as inputs
- `name`: a unique string identifier used to reference this node in edges

Below, `Add` computes `x_1 + x_2 → sum` and `Multiply` computes `scalar * x → product`.

In [ ]:
class Add(Node):
    def __init__(self, name):
        super().__init__(
            node=lambda X : {'sum' : X['x_1'] + X['x_2']},
            inputs={'x_1': None, 'x_2': None},
            outputs={'sum': None},
            name=name
        )

class Multiply(Node):
    def __init__(self, name):
        super().__init__(
            node=lambda X : {'product' : X['scalar'] * X['x']},
            inputs={'scalar': None, 'x': None},
            outputs={'product': None},
            name=name
        )

## Part 2 — Wiring Nodes into an HTG: `a * (b + c)`

Nodes are composed by calling `add_edge(source, target, rename={...}, features=[...])` on an
`HierarchalTensorGraph`. The special strings `"input"` and `"output"` refer to the HTG's
external interface.

Key `add_edge` parameters used here:

- **`rename`** — maps an output key from `source` to the input key expected by `target`.
  `Add` outputs `"sum"` but `Multiply` expects `"x"`, so `rename={'sum': 'x'}`.
- **`features`** — selects a subset of keys to pass across an edge. Only `"scalar"` is
  forwarded from `input` to `Multiply`; the `"x"` input arrives from `Add` via `rename`.

The computation graph is: `input → adder → multiplier → output`
with `input → multiplier` carrying only `scalar`.

In [ ]:
class AddMult(HierarchalTensorGraph):
    """Crest HTG to compute scalar * (x_1 + x_2)"""

    def __init__(self, name):
        super().__init__(name=name)

        adder = Add('adder')
        multiplier = Multiply('multiplier')

        self.add_edge('input', adder)
        # Only pass 'scalar' from input to multiplier; 'x' comes from adder via rename
        self.add_edge('input', multiplier, features=['scalar'])
        self.add_edge(adder, multiplier, rename={'sum': 'x'})
        self.add_edge(multiplier, 'output', rename={'product': 'add_mult'})

In [ ]:
import numpy as np

add_mult = AddMult('add_mult')
add_mult.draw()

In [ ]:
X = {'x_1' : 0.8, 'x_2' : 1.2, 'scalar' : 1.3}
add_mult(X)

## Part 3 — Hierarchical Composition: `exp(a * (b + c))`

An HTG can itself be used as a **node inside a larger HTG**. Here we place `AddMult` — the
two-node graph we just built — inside a new `AddMultExp` graph as a child node, creating a
two-level hierarchy:

```
AddMultExp
  ├─ add_mult  (AddMult HTG)
  │    ├─ adder       (Add node)
  │    └─ multiplier  (Multiply node)
  └─ exponentiate (Exp node)
```

Child HTGs are treated exactly like basenodes when wiring: their I/O keys must be matched
(or renamed) just as with any other node. The `draw()` and `print_edge_labels()` methods
support an `expand_nodes` argument to unfold child HTGs and inspect their internals.

In [ ]:
class Exp(Node):
    def __init__(self, name):
        super().__init__(
            node=lambda X : {'exp' : exp(X['x'])},
            inputs={'x': None},
            outputs={'exp': None},
            name=name
        )

class AddMultExp(HierarchalTensorGraph):

    def __init__(self):
        super().__init__(name='add_mult_exp')

        add_mult = AddMult('add_mult')
        exponentiate = Exp('exponentiate')

        # rename 'add_mult'->'x' on the add_mult->exp edge to match Exp's input;
        # rename 'exp'->'add_mult_exp' on the way out
        self.add_edge('input', add_mult)
        self.add_edge(add_mult, exponentiate, rename={'add_mult': 'x'})
        self.add_edge(exponentiate, 'output', rename={'exp': 'add_mult_exp'})

In [ ]:
add_mult_exp = AddMultExp()
add_mult_exp.print_edge_labels()

In [ ]:
print(add_mult_exp(X))
add_mult_exp.print_edge_labels()

### Visualizing the Graph

`draw()` renders the DAG. By default each HTG appears as a single collapsed node.
Pass `expand_nodes='<name>'` to unfold a specific child HTG, or `expand_nodes='all'` to
recursively expand every level of the hierarchy.

`print_edge_labels()` lists every edge with its source/target key names and any renames
applied. The same `expand_nodes` argument controls how deep the listing goes.

In [ ]:
add_mult_exp.draw()

In [ ]:
add_mult_exp.draw(expand_nodes='add_mult')

In [ ]:
add_mult_exp.draw(expand_nodes='all')

In [ ]:
add_mult_exp.print_edge_labels(expand_nodes='all')

We can also inspect just the child HTG's internal edges. This is useful when debugging a
specific sub-system without the clutter of the full hierarchy.

In [ ]:
add_mult.print_edge_labels()